# Lab 2: Pre-Trained Word Embeddings

**Course:** Deep Learning and Natural Language Processing (PhDAI-831-A01)  
**University:** University of the Cumberlands

## Part B: Model Implementation

### Step 1: Environment Setup

This section imports the libraries required for data processing, tokenization, numerical operations, progress tracking, and neural network development with PyTorch.

In [1]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\Users\dubey\Documents\UC_GitLab\.venv_global\Scripts\python.exe


In [2]:
# Core libraries
import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# NLP
import nltk
from nltk.tokenize import word_tokenize

# Utilities
from tqdm.auto import tqdm

print("PyTorch version :", torch.__version__)
print("NumPy version   :", np.__version__)
print("Pandas version  :", pd.__version__)
print("NLTK version    :", nltk.__version__)

PyTorch version : 2.10.0+cpu
NumPy version   : 1.26.4
Pandas version  : 2.3.3
NLTK version    : 3.10.3


### Step 2: NLTK Tokenizer Setup

NLTK's `word_tokenize()` function is used to split the AG News text into individual tokens. The required tokenizer resources are downloaded if they are not already available.

In [3]:
# Download required NLTK tokenizer resources
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dubey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dubey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
from pathlib import Path

DATA_DIR = Path("data")

print((DATA_DIR / "train.csv").exists())
print((DATA_DIR / "test.csv").exists())
print((DATA_DIR / "classes.txt").exists())

True
True
True


### Step 3: Load and Inspect the AG News Dataset

The AG News training and testing files are loaded using pandas. Each record contains a class label, a news title, and a news description. The class names are also loaded from `classes.txt`.

In [5]:
# Load AG News CSV files
train_df = pd.read_csv(
    DATA_DIR / "train.csv",
    header=None,
    names=["class", "title", "description"]
)

test_df = pd.read_csv(
    DATA_DIR / "test.csv",
    header=None,
    names=["class", "title", "description"]
)

# Load class names
with open(DATA_DIR / "classes.txt", "r", encoding="utf-8") as f:
    class_names = [line.strip() for line in f if line.strip()]

print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)

print("\nClasses:")
for i, class_name in enumerate(class_names, start=1):
    print(i, class_name)

Training shape: (120000, 3)
Testing shape : (7600, 3)

Classes:
1 World
2 Sports
3 Business
4 Sci/Tech


In [6]:
train_df.head()

,class,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


### Step 4: Clean Backslashes from the Text

The AG News dataset contains backslash characters in some titles and descriptions. These characters are replaced with spaces before tokenization so that words separated by backslashes are treated as separate tokens.

In [7]:
# Replace backslashes with spaces in the text columns
text_columns = ["title", "description"]

train_df[text_columns] = train_df[text_columns].replace(
    r"\\", " ", regex=True
)

test_df[text_columns] = test_df[text_columns].replace(
    r"\\", " ", regex=True
)

print("Backslash replacement completed.")

Backslash replacement completed.


In [8]:
train_df.head()

,class,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [9]:
# Verify that no backslashes remain
train_title_backslashes = train_df["title"].str.contains(r"\\", regex=True).sum()
train_desc_backslashes = train_df["description"].str.contains(r"\\", regex=True).sum()

test_title_backslashes = test_df["title"].str.contains(r"\\", regex=True).sum()
test_desc_backslashes = test_df["description"].str.contains(r"\\", regex=True).sum()

print("Train title backslashes      :", train_title_backslashes)
print("Train description backslashes:", train_desc_backslashes)
print("Test title backslashes       :", test_title_backslashes)
print("Test description backslashes :", test_desc_backslashes)

Train title backslashes      : 0
Train description backslashes: 0
Test title backslashes       : 0
Test description backslashes : 0


### Step 5: Tokenize the Title and Description

The cleaned title and description fields are tokenized using NLTK's `word_tokenize()` function. The text is converted to lowercase before tokenization to improve consistency and increase the likelihood that tokens will match entries in the pre-trained GloVe vocabulary.

In [10]:
# Enable tqdm progress bars for pandas operations
tqdm.pandas()

# Tokenize title and description columns
train_df["title_tokens"] = train_df["title"].progress_apply(
    lambda text: word_tokenize(str(text).lower())
)

train_df["description_tokens"] = train_df["description"].progress_apply(
    lambda text: word_tokenize(str(text).lower())
)

test_df["title_tokens"] = test_df["title"].progress_apply(
    lambda text: word_tokenize(str(text).lower())
)

test_df["description_tokens"] = test_df["description"].progress_apply(
    lambda text: word_tokenize(str(text).lower())
)

print("Tokenization completed.")

  0%|          | 0/120000 [00:00<?, ?it/s]

  0%|          | 0/120000 [00:00<?, ?it/s]

  0%|          | 0/7600 [00:00<?, ?it/s]

  0%|          | 0/7600 [00:00<?, ?it/s]

Tokenization completed.


In [11]:
print("Original title:")
print(train_df.loc[0, "title"])

print("\nTitle tokens:")
print(train_df.loc[0, "title_tokens"])

print("\nOriginal description:")
print(train_df.loc[0, "description"])

print("\nDescription tokens:")
print(train_df.loc[0, "description_tokens"])

Original title:
Wall St. Bears Claw Back Into the Black (Reuters)

Title tokens:
['wall', 'st.', 'bears', 'claw', 'back', 'into', 'the', 'black', '(', 'reuters', ')']

Original description:
Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.

Description tokens:
['reuters', '-', 'short-sellers', ',', 'wall', 'street', "'s", 'dwindling', 'band', 'of', 'ultra-cynics', ',', 'are', 'seeing', 'green', 'again', '.']


### Step 6: Combine Title and Description Tokens

The tokenized title and description are combined into a single token sequence for each news article. This combined sequence will later be mapped to pre-trained GloVe word embeddings.

In [12]:
# Combine title and description tokens
train_df["tokens"] = (
    train_df["title_tokens"] + train_df["description_tokens"]
)

test_df["tokens"] = (
    test_df["title_tokens"] + test_df["description_tokens"]
)

print("Combined token sequences created.")

Combined token sequences created.


In [13]:
print("Combined tokens for first training example:")
print(train_df.loc[0, "tokens"])

print("\nNumber of tokens:")
print(len(train_df.loc[0, "tokens"]))

Combined tokens for first training example:
['wall', 'st.', 'bears', 'claw', 'back', 'into', 'the', 'black', '(', 'reuters', ')', 'reuters', '-', 'short-sellers', ',', 'wall', 'street', "'s", 'dwindling', 'band', 'of', 'ultra-cynics', ',', 'are', 'seeing', 'green', 'again', '.']

Number of tokens:
28


In [14]:
try:
    import gensim
    from gensim.models import KeyedVectors

    print("Gensim version:", gensim.__version__)
    print("Gensim is ready.")
except ImportError:
    print("Gensim is not installed.")

Gensim version: 4.4.0
Gensim is ready.


### Step 7A: Install Gensim

Gensim is used to load and work with pre-trained GloVe word embeddings. It is installed into the same Python environment used by this notebook.

In [15]:
import sys
!{sys.executable} -m pip install gensim


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import gensim
from gensim.models import KeyedVectors

print("Gensim version:", gensim.__version__)
print("Gensim is ready.")

Gensim version: 4.4.0
Gensim is ready.


### Step 7B: Load Pre-Trained GloVe Word Embeddings

Pre-trained GloVe word embeddings are loaded using Gensim. The 300-dimensional GloVe model provides a dense vector representation for each word based on statistical relationships learned from a large text corpus.

In [17]:
import gensim.downloader as api

# Load pre-trained 300-dimensional GloVe embeddings
glove = api.load("glove-wiki-gigaword-300")

print("GloVe embeddings loaded.")
print("Vocabulary size :", len(glove))
print("Embedding size  :", glove.vector_size)

GloVe embeddings loaded.
Vocabulary size : 400000
Embedding size  : 300


### Step 8: Estimate GloVe Vocabulary Coverage

To determine how well the tokenized AG News text matches the pre-trained GloVe vocabulary, the training tokens are compared with the GloVe vocabulary. Coverage is measured both across all token occurrences and across unique tokens.

In [18]:
from collections import Counter
from itertools import chain

# Count all tokens in the training dataset
token_counts = Counter(chain.from_iterable(train_df["tokens"]))

total_tokens = sum(token_counts.values())
unique_tokens = len(token_counts)

# Identify tokens that exist in the GloVe vocabulary
matched_unique_tokens = [
    token for token in token_counts
    if token in glove.key_to_index
]

matched_token_count = sum(
    token_counts[token] for token in matched_unique_tokens
)

# Calculate coverage percentages
token_coverage = (matched_token_count / total_tokens) * 100
unique_coverage = (len(matched_unique_tokens) / unique_tokens) * 100

print(f"Total token occurrences : {total_tokens:,}")
print(f"Matched token occurrences: {matched_token_count:,}")
print(f"Token coverage          : {token_coverage:.2f}%")

print()

print(f"Unique tokens           : {unique_tokens:,}")
print(f"Matched unique tokens   : {len(matched_unique_tokens):,}")
print(f"Unique-token coverage   : {unique_coverage:.2f}%")

Total token occurrences : 5,275,934
Matched token occurrences: 5,211,909
Token coverage          : 98.79%

Unique tokens           : 88,124
Matched unique tokens   : 64,652
Unique-token coverage   : 73.36%


#### Inspect Out-of-Vocabulary Tokens

Tokens that are not present in the GloVe vocabulary are considered out-of-vocabulary (OOV). Examining the most frequent OOV tokens helps identify why some tokenized words cannot be mapped to pre-trained embeddings.

In [19]:
# Identify out-of-vocabulary (OOV) tokens
oov_tokens = {
    token: count
    for token, count in token_counts.items()
    if token not in glove.key_to_index
}

# Show the most frequent OOV tokens
top_oov = sorted(
    oov_tokens.items(),
    key=lambda x: x[1],
    reverse=True
)[:30]

print("Number of unique OOV tokens:", len(oov_tokens))
print("\nTop 30 most frequent OOV tokens:")

for token, count in top_oov:
    print(f"{token:<30} {count:,}")

Number of unique OOV tokens: 23472

Top 30 most frequent OOV tokens:
/b                             2,984
href=                          2,119
/a                             2,117
//www.investor.reuters.com/fullquote.aspx 1,813
target=/stocks/quickinfo/fullquote 1,813
/p                             537
newsfactor                     510
cbs.mw                         471
color=                         431
/font                          417
face=                          416
size=                          416
666666                         416
//www.reuters.co.uk/financequotelookup.jhtml 242
qtype=sym                      242
infotype=info                  242
qcat=news                      242
maccentral                     214
-the                           213
/strong                        184
techweb                        140
siliconvalley.com              134
cnn/money                      134
newratings.com                 134
-washingtonpost.com            117
2004.            

### Step 9: Add Unknown and Padding Embeddings

Two additional embeddings are added to the pre-trained GloVe vocabulary. The `[UNK]` embedding represents tokens that are not available in the GloVe vocabulary and is initialized using the mean of the existing GloVe vectors. The `[PAD]` embedding is initialized with zeros and will be used to pad token sequences to a common length for mini-batch processing.

In [20]:
# Special tokens
UNK_TOKEN = "[UNK]"
PAD_TOKEN = "[PAD]"

# Create the special embeddings
unk_embedding = glove.vectors.mean(axis=0)
pad_embedding = np.zeros(
    glove.vector_size,
    dtype=glove.vectors.dtype
)

# Add them only if they have not already been added
if UNK_TOKEN not in glove.key_to_index:
    glove.add_vector(UNK_TOKEN, unk_embedding)

if PAD_TOKEN not in glove.key_to_index:
    glove.add_vector(PAD_TOKEN, pad_embedding)

# Obtain their token IDs
unk_id = glove.key_to_index[UNK_TOKEN]
pad_id = glove.key_to_index[PAD_TOKEN]

print("UNK token ID       :", unk_id)
print("PAD token ID       :", pad_id)
print("Vocabulary size    :", len(glove))
print("Embedding dimension:", glove.vector_size)
print("PAD vector sum     :", glove[PAD_TOKEN].sum())

c:\Users\dubey\Documents\UC_GitLab\.venv_global\Lib\site-packages\gensim\models\keyedvectors.py:551: UserWarning: Adding single vectors to a KeyedVectors which grows by one each time can be costly. Consider adding in batches or preallocating to the required size.
  warnings.warn(


UNK token ID       : 400000
PAD token ID       : 400001
Vocabulary size    : 400002
Embedding dimension: 300
PAD vector sum     : 0.0


### Step 10: Create Training and Development Partitions

The original AG News training dataset is divided into training and development partitions. The development partition is used to evaluate the model after each training epoch, while the original test dataset is reserved for final evaluation.

In [21]:
# Reproducibility
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [22]:
from sklearn.model_selection import train_test_split

train_part_df, dev_df = train_test_split(
    train_df,
    test_size=0.20,
    random_state=SEED,
    stratify=train_df["class"]
)

# Reset indices after splitting
train_part_df = train_part_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training partition   :", train_part_df.shape)
print("Development partition:", dev_df.shape)
print("Testing partition    :", test_df.shape)

print("\nTraining class counts:")
print(train_part_df["class"].value_counts().sort_index())

print("\nDevelopment class counts:")
print(dev_df["class"].value_counts().sort_index())

Training partition   : (96000, 6)
Development partition: (24000, 6)
Testing partition    : (7600, 6)

Training class counts:
class
1    24000
2    24000
3    24000
4    24000
Name: count, dtype: int64

Development class counts:
class
1    6000
2    6000
3    6000
4    6000
Name: count, dtype: int64


### Step 11: Build the Training Vocabulary

A vocabulary is created from the training partition using tokens that occur more than 10 times. Infrequent tokens are treated as unknown during token-ID conversion. This reduces the influence of very rare or noisy tokens while retaining the most useful vocabulary for classification.

In [23]:
# Minimum frequency threshold
threshold = 10

# Count token frequencies in the training partition
train_token_frequencies = (
    train_part_df["tokens"]
    .explode()
    .value_counts()
)

# Keep tokens occurring more than the threshold
vocabulary = set(
    train_token_frequencies[
        train_token_frequencies > threshold
    ].index
)

print(f"Frequency threshold : > {threshold}")
print(f"Training vocabulary : {len(vocabulary):,} tokens")

Frequency threshold : > 10
Training vocabulary : 17,516 tokens


### Step 12: Convert Tokens to Padded Token-ID Sequences

Each token is converted to its corresponding GloVe vocabulary ID. Tokens that are infrequent or absent from GloVe are mapped to the `[UNK]` ID. Because mini-batches require sequences with consistent dimensions, shorter sequences are padded using the `[PAD]` token.

In [24]:
# Inspect token-sequence lengths
train_max_tokens = train_part_df["tokens"].map(len).max()
dev_max_tokens = dev_df["tokens"].map(len).max()
test_max_tokens = test_df["tokens"].map(len).max()

train_avg_tokens = train_part_df["tokens"].map(len).mean()
dev_avg_tokens = dev_df["tokens"].map(len).mean()
test_avg_tokens = test_df["tokens"].map(len).mean()

print(f"Training - maximum tokens : {train_max_tokens}")
print(f"Training - average tokens : {train_avg_tokens:.2f}")

print(f"\nDevelopment - maximum tokens : {dev_max_tokens}")
print(f"Development - average tokens : {dev_avg_tokens:.2f}")

print(f"\nTesting - maximum tokens : {test_max_tokens}")
print(f"Testing - average tokens : {test_avg_tokens:.2f}")

Training - maximum tokens : 249
Training - average tokens : 43.99

Development - maximum tokens : 231
Development - average tokens : 43.88

Testing - maximum tokens : 165
Testing - average tokens : 43.69


#### Convert Tokens to IDs and Pad the Sequences

Each token is mapped to its corresponding GloVe vocabulary ID. Tokens that either occur too infrequently in the training vocabulary or are unavailable in GloVe are mapped to the `[UNK]` token. Each sequence is padded with the `[PAD]` token so that all examples have the same length.

In [25]:
# Use the maximum sequence length found in the training partition
MAX_LENGTH = train_max_tokens

def token_to_id(token):
    """
    Convert a token to its GloVe vocabulary ID.
    Rare tokens and tokens absent from GloVe are mapped to UNK.
    """
    if token in vocabulary and token in glove.key_to_index:
        return glove.key_to_index[token]
    
    return unk_id


def dataframe_to_arrays(df, max_length):
    """
    Convert tokenized documents into:
      X -> 2-D NumPy array of padded token IDs
      y -> 1-D NumPy array of class labels
    """
    
    # Start with an array containing only PAD IDs
    X = np.full(
        (len(df), max_length),
        pad_id,
        dtype=np.int64
    )

    # Convert each document to token IDs
    for i, tokens in enumerate(
        tqdm(df["tokens"], desc="Converting documents")
    ):
        token_ids = [
            token_to_id(token)
            for token in tokens[:max_length]
        ]

        X[i, :len(token_ids)] = token_ids

    # AG News labels are 1-4.
    # PyTorch CrossEntropyLoss expects 0-3.
    y = df["class"].to_numpy(dtype=np.int64) - 1

    return X, y

In [26]:
sample_tokens = train_part_df.loc[0, "tokens"]

sample_ids = [
    token_to_id(token)
    for token in sample_tokens
]

print("Number of tokens :", len(sample_tokens))
print("Number of IDs    :", len(sample_ids))

print("\nFirst 15 tokens:")
print(sample_tokens[:15])

print("\nFirst 15 token IDs:")
print(sample_ids[:15])

Number of tokens : 40
Number of IDs    : 40

First 15 tokens:
['clijsters', 'unsure', 'about', 'latest', 'injury', ',', 'says', 'hewitt', 'tokyo', '(', 'reuters', ')', '-', 'belgian', 'kim']

First 15 token IDs:
[12047, 13642, 59, 993, 1553, 1, 210, 7839, 1363, 23, 10851, 24, 11, 4072, 2017]


#### Create NumPy Arrays for Model Input

The tokenized documents are converted into two-dimensional NumPy arrays containing padded token IDs. The corresponding class labels are stored in one-dimensional NumPy arrays. Since AG News labels are numbered 1 through 4, one is subtracted so that the PyTorch target classes range from 0 through 3.

In [27]:
# Convert each partition to padded token-ID arrays and label arrays

X_train, y_train = dataframe_to_arrays(
    train_part_df,
    MAX_LENGTH
)

X_dev, y_dev = dataframe_to_arrays(
    dev_df,
    MAX_LENGTH
)

X_test, y_test = dataframe_to_arrays(
    test_df,
    MAX_LENGTH
)

print("\nArray creation completed.")

Converting documents:   0%|          | 0/96000 [00:00<?, ?it/s]

Converting documents:   0%|          | 0/24000 [00:00<?, ?it/s]

Converting documents:   0%|          | 0/7600 [00:00<?, ?it/s]


Array creation completed.


In [28]:
print("Training:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nDevelopment:")
print("X_dev shape  :", X_dev.shape)
print("y_dev shape  :", y_dev.shape)

print("\nTesting:")
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)

print("\nLabel range:")
print("Training labels:", y_train.min(), "to", y_train.max())
print("Dev labels     :", y_dev.min(), "to", y_dev.max())
print("Test labels    :", y_test.min(), "to", y_test.max())

Training:
X_train shape: (96000, 249)
y_train shape: (96000,)

Development:
X_dev shape  : (24000, 249)
y_dev shape  : (24000,)

Testing:
X_test shape : (7600, 249)
y_test shape : (7600,)

Label range:
Training labels: 0 to 3
Dev labels     : 0 to 3
Test labels    : 0 to 3


### Step 13: Create PyTorch DataLoaders

The NumPy arrays are converted to PyTorch tensors and organized into `TensorDataset` objects. `DataLoader` objects divide the datasets into mini-batches for efficient model training and evaluation. The training data are shuffled at the beginning of each epoch, while the development and testing data remain in a fixed order.

In [29]:
# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train).long()
y_train_tensor = torch.from_numpy(y_train).long()

X_dev_tensor = torch.from_numpy(X_dev).long()
y_dev_tensor = torch.from_numpy(y_dev).long()

X_test_tensor = torch.from_numpy(X_test).long()
y_test_tensor = torch.from_numpy(y_test).long()

# Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
dev_dataset = TensorDataset(X_dev_tensor, y_dev_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Batch size used for training and evaluation
BATCH_SIZE = 500

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training examples   :", len(train_dataset))
print("Development examples:", len(dev_dataset))
print("Testing examples    :", len(test_dataset))

print("\nTraining batches   :", len(train_loader))
print("Development batches:", len(dev_loader))
print("Testing batches    :", len(test_loader))

Training examples   : 96000
Development examples: 24000
Testing examples    : 7600

Training batches   : 192
Development batches: 48
Testing batches    : 16


In [30]:
X_batch, y_batch = next(iter(train_loader))

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)
print("X dtype      :", X_batch.dtype)
print("y dtype      :", y_batch.dtype)

X batch shape: torch.Size([500, 249])
y batch shape: torch.Size([500])
X dtype      : torch.int64
y dtype      : torch.int64


### Step 14: Construct the Feed-Forward Neural Network

The model uses the pre-trained 300-dimensional GloVe vectors as its embedding layer. For each document, the embeddings of all non-padding tokens are averaged to produce a single 300-dimensional document representation.

This representation is passed through a two-layer feed-forward neural network. The first linear layer maps the 300-dimensional input to 50 hidden units, followed by a ReLU activation and dropout. The second linear layer maps the hidden representation to the four AG News output classes.

In [31]:
# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [32]:
class GloveFFNN(nn.Module):
    def __init__(
        self,
        vectors,
        pad_id,
        hidden_dim,
        output_dim,
        dropout
    ):
        super().__init__()

        self.padding_idx = pad_id

        # Pre-trained GloVe embedding layer
        self.embedding = nn.Embedding.from_pretrained(
            vectors,
            freeze=True,
            padding_idx=pad_id
        )

        embedding_dim = vectors.shape[1]

        # Two-layer feed-forward neural network
        self.layers = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):

        # Identify real tokens (not padding)
        not_padding = (x != self.padding_idx)

        # Number of real tokens in each document
        lengths = not_padding.sum(dim=1).clamp(min=1)

        # Convert token IDs to GloVe vectors
        embeddings = self.embedding(x)

        # Mean embedding for each document
        document_vectors = (
            embeddings.sum(dim=1)
            / lengths.unsqueeze(1)
        )

        # Feed document representation through FFNN
        logits = self.layers(document_vectors)

        return logits

In [33]:
# Model hyperparameters
HIDDEN_DIM = 50
OUTPUT_DIM = len(class_names)
DROPOUT = 0.1

# Convert GloVe vectors to a PyTorch tensor
embedding_weights = torch.from_numpy(glove.vectors)

# Initialize model
model = GloveFFNN(
    vectors=embedding_weights,
    pad_id=pad_id,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    dropout=DROPOUT
).to(device)

print(model)

GloveFFNN(
  (embedding): Embedding(400002, 300, padding_idx=400001)
  (layers): Sequential(
    (0): Dropout(p=0.1, inplace=False)
    (1): Linear(in_features=300, out_features=50, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=50, out_features=4, bias=True)
  )
)


In [34]:
X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)

with torch.no_grad():
    output = model(X_batch)

print("Input batch shape :", X_batch.shape)
print("Model output shape:", output.shape)

Input batch shape : torch.Size([500, 249])
Model output shape: torch.Size([500, 4])


### Step 15: Configure the Training Procedure

Cross-entropy loss is used because the task is a four-class classification problem. The Adam optimizer updates the trainable parameters of the feed-forward neural network during backpropagation. The pre-trained GloVe embedding weights remain frozen during training.

In [35]:
# Training hyperparameters
LEARNING_RATE = 0.001
NUM_EPOCHS = 10

# Multi-class classification loss
criterion = nn.CrossEntropyLoss()

# Adam optimizer
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)

print("Loss function :", criterion)
print("Optimizer     : Adam")
print("Learning rate:", LEARNING_RATE)
print("Epochs        :", NUM_EPOCHS)

Loss function : CrossEntropyLoss()
Optimizer     : Adam
Learning rate: 0.001
Epochs        : 10


### Step 16: Train and Validate the Model

The model is trained using mini-batch gradient descent with the Adam optimizer. During each training batch, the model performs a forward pass, computes cross-entropy loss, performs backpropagation, and updates the trainable network parameters.

After every epoch, the model is evaluated on the development partition. Development loss and accuracy are recorded to monitor how well the model generalizes to unseen examples.

In [36]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in tqdm(
        data_loader,
        desc="Training",
        leave=False
    ):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch)

        # Compute loss
        loss = criterion(logits, y_batch)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate statistics
        batch_size = y_batch.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            logits.argmax(dim=1) == y_batch
        ).sum().item()

        total_examples += batch_size

    average_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return average_loss, accuracy

In [37]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():

        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            logits = model(X_batch)

            # Compute loss
            loss = criterion(logits, y_batch)

            batch_size = y_batch.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (
                logits.argmax(dim=1) == y_batch
            ).sum().item()

            total_examples += batch_size

    average_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return average_loss, accuracy

In [38]:
history = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    dev_loss, dev_accuracy = evaluate_model(
        model,
        dev_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "dev_loss": dev_loss,
        "dev_accuracy": dev_accuracy
    })

    print(
        f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Dev Loss: {dev_loss:.4f} | "
        f"Dev Acc: {dev_accuracy:.4f}"
    )

Training:   0%|          | 0/192 [00:00<?, ?it/s]

Epoch  1/10 | Train Loss: 0.6382 | Train Acc: 0.8288 | Dev Loss: 0.3638 | Dev Acc: 0.8849


Training:   0%|          | 0/192 [00:00<?, ?it/s]

Epoch  2/10 | Train Loss: 0.3644 | Train Acc: 0.8801 | Dev Loss: 0.3236 | Dev Acc: 0.8953


Training:   0%|          | 0/192 [00:00<?, ?it/s]

...

Epoch  8/10 | Train Loss: 0.3035 | Train Acc: 0.8947 | Dev Loss: 0.2779 | Dev Acc: 0.9042


Training:   0%|          | 0/192 [00:00<?, ?it/s]

Epoch  9/10 | Train Loss: 0.3009 | Train Acc: 0.8952 | Dev Loss: 0.2753 | Dev Acc: 0.9058


Training:   0%|          | 0/192 [00:00<?, ?it/s]

Epoch 10/10 | Train Loss: 0.2985 | Train Acc: 0.8942 | Dev Loss: 0.2722 | Dev Acc: 0.9052


### Step 17: Evaluate the Model on the Test Partition

After training, the final model is evaluated on the held-out AG News testing partition. Test loss and accuracy are calculated, followed by precision, recall, and F1-score for each of the four news categories.

In [39]:
# Evaluate final model on the test partition
test_loss, test_accuracy = evaluate_model(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss    : 0.2862
Test Accuracy: 0.9007


In [40]:
from sklearn.metrics import classification_report

all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        predictions = logits.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            y_batch.cpu().numpy()
        )

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)

              precision    recall  f1-score   support

       World     0.9239    0.8879    0.9055      1900
      Sports     0.9522    0.9742    0.9631      1900
    Business     0.8602    0.8579    0.8590      1900
    Sci/Tech     0.8667    0.8826    0.8746      1900

    accuracy                         0.9007      7600
   macro avg     0.9007    0.9007    0.9005      7600
weighted avg     0.9007    0.9007    0.9005      7600



## Part A: Feed-Forward Neural Networks and Word Embeddings

A text classification model using pre-trained word embeddings and a feed-forward neural network converts textual documents into numerical representations that can be processed by a neural network. In this implementation, the AG News Topic Classification dataset is used to classify news articles into four categories: World, Sports, Business, and Sci/Tech.

The text is first cleaned and tokenized using NLTK's `word_tokenize()` function. Each token is then matched with a pre-trained 300-dimensional GloVe word embedding. GloVe represents words as dense numerical vectors learned from large text corpora, allowing words that occur in similar linguistic contexts to have related vector representations (Pennington et al., 2014). Tokens that are unavailable in the GloVe vocabulary are mapped to an `[UNK]` embedding, while a zero-valued `[PAD]` embedding is used to make document sequences a consistent length.

Each news article is represented by the mean of the embeddings of its non-padding tokens. This converts a variable-length sequence of word vectors into a single 300-dimensional document representation. The representation is then passed through a two-layer feed-forward neural network. The first linear layer maps the 300 input features to 50 hidden units. A ReLU activation introduces nonlinearity, and dropout is applied to reduce overfitting. The second linear layer produces four output scores corresponding to the four AG News classes.

The network is trained using cross-entropy loss and the Adam optimizer. During each training iteration, a forward pass produces class scores, the loss measures the difference between the predictions and correct labels, and backpropagation calculates gradients used to update the trainable neural-network parameters. The pre-trained GloVe embeddings remain frozen so that the model retains the semantic information learned during GloVe pre-training.

The model is evaluated on a separate development partition after each epoch using loss and classification accuracy. After training is complete, performance is measured on the held-out test partition using accuracy, precision, recall, and F1-score. This approach combines the semantic information contained in pre-trained word embeddings with the learning capability of a feed-forward neural network for efficient multi-class text classification.

## Part C: Analysis

### Explanation of the Part B Implementation

The implementation begins by importing NumPy, pandas, PyTorch, NLTK, and tqdm. NumPy is used for numerical arrays, pandas manages the AG News dataset, PyTorch provides the neural-network framework, NLTK performs tokenization, and tqdm displays progress during preprocessing and training.

The AG News `train.csv` and `test.csv` files are loaded with `pd.read_csv()`. Because the CSV files do not contain column headers, the columns are explicitly named `class`, `title`, and `description`. The `classes.txt` file provides the four class names: World, Sports, Business, and Sci/Tech.

Backslash characters in the title and description columns are replaced with spaces using pandas `replace()` with `regex=True`. This prevents words separated by backslashes in the raw dataset from being incorrectly treated as a single token.

The title and description fields are converted to lowercase and tokenized with NLTK's `word_tokenize()`. Lowercasing improves consistency when matching words with the GloVe vocabulary. The title and description tokens are then combined so that each news article is represented by one token sequence.

Pre-trained 300-dimensional GloVe embeddings are loaded through Gensim. The GloVe vocabulary contains 400,000 pre-trained word vectors. Vocabulary coverage is estimated by checking whether each AG News token occurs in the GloVe vocabulary. The training data achieved 98.79% coverage across all token occurrences and 73.36% coverage across unique tokens. This indicates that most frequently occurring words in the dataset have a corresponding pre-trained embedding.

Two additional embeddings are added. `[UNK]` represents rare or out-of-vocabulary tokens and is initialized using the average GloVe vector. `[PAD]` is represented by a vector of zeros and is used to make sequences equal in length. A training vocabulary is created using words occurring more than 10 times, producing 17,516 retained tokens.

The original training dataset is divided into 96,000 training examples and 24,000 development examples using a stratified split. Stratification preserves the equal distribution of the four classes. The original 7,600 examples remain reserved for final testing.

Each document is converted into GloVe token IDs. Rare words and words unavailable in GloVe are assigned the `[UNK]` ID. All documents are padded to the maximum training sequence length of 249 tokens using the `[PAD]` ID. This produces two-dimensional NumPy arrays for the document token IDs and one-dimensional arrays for the gold class labels. Because PyTorch's `CrossEntropyLoss` expects class indices beginning at zero, the original AG News labels 1–4 are converted to 0–3.

The NumPy arrays are converted into PyTorch tensors and wrapped in `TensorDataset` and `DataLoader` objects. A batch size of 500 produces 192 training batches, 48 development batches, and 16 testing batches.

The neural network uses `nn.Embedding.from_pretrained()` to load the GloVe vectors. The embedding layer is frozen so that the pre-trained vectors are not modified during training. For each article, the embeddings of the non-padding tokens are averaged to create one 300-dimensional document representation. This vector passes through a linear layer with 50 hidden units, a ReLU activation, dropout, and a final linear layer with four output units.

Training uses `CrossEntropyLoss()` and the Adam optimizer with a learning rate of 0.001. During each batch, `optimizer.zero_grad()` clears previous gradients, the forward pass produces class scores, the loss is calculated, `loss.backward()` computes gradients through backpropagation, and `optimizer.step()` updates the trainable network parameters. Development loss and accuracy are calculated after every epoch with gradient computation disabled using `torch.no_grad()`.

For final evaluation, the trained model predicts the classes of the held-out test examples. Accuracy provides the overall percentage of correct predictions, while precision, recall, and F1-score provide class-specific measures of model performance.

### Evaluation Results and Interpretation

The feed-forward neural network achieved a final test loss of **0.2862** and a test accuracy of **0.9007**, meaning that approximately **90.07% of the 7,600 unseen AG News test articles were classified correctly**. The macro-average F1-score and weighted-average F1-score were both approximately **0.9005**, indicating strong and relatively balanced performance across the four classes.

During training, model performance improved steadily. Development accuracy increased from **88.49% after the first epoch** to **90.52% after the tenth epoch**, while development loss decreased from **0.3638 to 0.2722**. The highest development accuracy was **90.58% at epoch 9**. The final test accuracy of 90.07% is close to the development performance, suggesting that the model generalized well to unseen data rather than simply memorizing the training examples.

The **Sports** category produced the strongest results, with a precision of **0.9522**, recall of **0.9742**, and F1-score of **0.9631**. The high recall indicates that the model correctly identified most Sports articles in the test dataset. Sports articles often contain distinctive terminology associated with teams, competitions, players, games, and sporting events, which can make the category easier to distinguish using word embeddings.

The **World** category also performed strongly, achieving a precision of **0.9239**, recall of **0.8879**, and F1-score of **0.9055**. Its precision was higher than its recall, meaning that predictions labeled as World were generally reliable, although some actual World articles were classified into another category.

The **Business** category had a precision of **0.8602**, recall of **0.8579**, and F1-score of **0.8590**, while **Sci/Tech** achieved a precision of **0.8667**, recall of **0.8826**, and F1-score of **0.8746**. These two categories were more difficult for the model than Sports and World. One possible reason is that Business and Sci/Tech articles can contain overlapping vocabulary related to companies, products, investments, technology firms, markets, and corporate activity. Because the model represents an entire article by averaging its word embeddings, some category-specific distinctions may be reduced.

The results also illustrate both the strength and limitation of the mean-embedding approach. Pre-trained GloVe vectors provide meaningful semantic representations while reducing each document to only 300 numerical features. However, averaging the vectors ignores word order and treats the document largely as a collection of semantic word representations. Consequently, sentences with different structures can receive similar document representations.

Overall, the model demonstrated that pre-trained GloVe embeddings combined with a relatively simple two-layer feed-forward neural network can provide effective multi-class text classification. Achieving approximately 90% accuracy with frozen pre-trained embeddings also shows that useful linguistic information learned from a large external corpus can transfer successfully to the AG News classification task.

## Conclusion

This lab implemented a multi-class text classification model using pre-trained GloVe word embeddings and a two-layer feed-forward neural network. The AG News dataset was cleaned, tokenized, converted into GloVe token IDs, padded to a consistent sequence length, and divided into training, development, and testing partitions.

The GloVe vocabulary provided strong coverage of the dataset, matching **98.79% of all token occurrences**. The neural network used averaged 300-dimensional GloVe embeddings as document representations and classified articles into World, Sports, Business, and Sci/Tech categories.

After 10 training epochs, the model achieved a final development accuracy of **90.52%** and a test accuracy of **90.07%**, with a macro F1-score of **90.05%**. Sports produced the strongest class-level performance, while Business and Sci/Tech were somewhat more difficult to distinguish.

Overall, the results demonstrate that pre-trained word embeddings can transfer useful semantic information to a downstream text classification task even when the embedding weights remain frozen. At the same time, averaging word embeddings removes word-order information, illustrating why more advanced sequence-based and contextual models may further improve text classification performance.

## References

Pennington, J., Socher, R., & Manning, C. D. (2014). GloVe: Global vectors for word representation. *Proceedings of the 2014 Conference on Empirical Methods in Natural Language Processing (EMNLP)*, 1532–1543. https://doi.org/10.3115/v1/D14-1162

Zhang, X., Zhao, J., & LeCun, Y. (2015). Character-level convolutional networks for text classification. *Advances in Neural Information Processing Systems, 28*, 649–657.

PyTorch. (n.d.). *PyTorch documentation*. https://pytorch.org/docs/stable/

NLTK Project. (n.d.). *Natural Language Toolkit documentation*. https://www.nltk.org/

Řehůřek, R., & Sojka, P. (2010). Software framework for topic modelling with large corpora. *Proceedings of the LREC 2010 Workshop on New Challenges for NLP Frameworks*, 45–50.

Surdeanu, M., & Valenzuela-Escárcega, M. A. *Deep learning for natural language processing*. Course/reference materials.

CLULab. (n.d.). *Text classification notebook (Chapter 9)*. GitHub repository: `clulab/gentlenlp`.

## AI Use Disclosure

ChatGPT (OpenAI) was used as a learning and development aid during this lab. It was used to help explain concepts related to pre-trained word embeddings, GloVe, feed-forward neural networks, PyTorch, data preprocessing, model training, and evaluation. It also assisted in organizing the notebook, developing code examples, interpreting model results, and improving the clarity of written explanations.

All code was executed in the author's local Python environment, and the resulting dataset dimensions, vocabulary coverage, training metrics, development metrics, and testing results were independently verified from the actual notebook outputs. The final implementation and analysis were reviewed against the lab requirements and the referenced course notebook. The author remains responsible for the accuracy, originality, and integrity of the submitted work.